# LLM Improvements

In [31]:
from google import genai
from dotenv import load_dotenv
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import os

load_dotenv()
os.chdir(r"C:\Users\ACER\NamTran\GED\B2\Semester II\ML\Mood dectection")

df = pd.read_csv("data/processed/cleaned_data.csv")

def assign_mood(row):
    if row['valence'] >= 0.5 and row['energy'] >= 0.5:
        return 'Happy'
    elif row['valence'] < 0.5 and row['energy'] >= 0.5:
        return 'Angry'
    elif row['valence'] >= 0.5 and row['energy'] < 0.5:
        return 'Calm'
    else:
        return 'Sad'

df['mood'] = df.apply(assign_mood, axis=1)
print("Data loaded:", df.shape)
print(df['mood'].value_counts())

Data loaded: (81343, 17)
mood
Happy    29525
Angry    27800
Sad      17572
Calm      6446
Name: count, dtype: int64


In [32]:
FEATURES = ['danceability', 'loudness', 'tempo', 'acousticness',
            'speechiness', 'instrumentalness', 'liveness',
            'duration_ms', 'explicit', 'popularity']

X = df[FEATURES]
y = df['mood']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

baseline = LogisticRegression(max_iter=1000, random_state=42)
baseline.fit(X_train_s, y_train)
y_pred_base = baseline.predict(X_test_s)

f1_base  = f1_score(y_test, y_pred_base, average="macro")
acc_base = accuracy_score(y_test, y_pred_base)

print(f"Baseline accuracy:  {acc_base:.4f}")
print(f"Baseline macro-F1:  {f1_base:.4f}")

Baseline accuracy:  0.6368
Baseline macro-F1:  0.5806


## 1, Load the API key from google AI studio

In [7]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
print("Connected!")

Connected!


## 2, Ask Gemini

In [30]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

PROMPT = """I'm building a music mood classifier (Happy/Angry/Calm/Sad) 
using Spotify audio features. I cannot use valence or energy.

Available: danceability, loudness, tempo, acousticness, speechiness,
instrumentalness, liveness, duration_ms, explicit, popularity.

Suggest 3 derived features that could help. One sentence each. No preamble."""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=PROMPT
)
print(response.text)

os.makedirs("results", exist_ok=True)
with open("results/llm_prompts.md", "w") as f:
    f.write("## Prompt\n\n" + PROMPT)
    f.write("\n\n## Gemini Response\n\n" + response.text)

print("\nSaved to results/llm_prompts.md")

1.  **Vocal Emotionality Score**: A composite of speechiness, explicit content, and inverse instrumentalness, reflecting the direct and often intense conveyance of emotion through prominent vocals.
2.  **Tranquil Soundscape Index**: Integrates high acousticness and instrumentalness with lower tempo and loudness, indicating a natural, peaceful, or contemplative musical character.
3.  **Rhythmic Character Balance**: Combines danceability and tempo while inversely considering speechiness, highlighting the relative prominence of physical groove versus spoken delivery.

Saved to results/llm_prompts.md


## 3, Save the results and prompts (Optional)

In [14]:
import os

os.makedirs("results", exist_ok=True)

with open("results/llm_prompts.md", "w") as f:
    f.write("## Prompt\n\n")
    f.write(PROMPT)
    f.write("\n\n## Raw Response\n\n")
    f.write(response.text)

print("Saved to results/llm_prompts.md")

Saved to results/llm_prompts.md


## 4, Apply the improvements

In [33]:
# Add the new feature Gemini suggested
df["dynamic_intensity"] = df["tempo"] * (df["loudness"] + 60)

FEATURES_IMPROVED = FEATURES + ["dynamic_intensity"]

X_imp = df[FEATURES_IMPROVED]

# Use the SAME split — same random_state=42
X_train_imp, X_test_imp, y_train_imp, y_test_imp = train_test_split(
    X_imp, y, test_size=0.2, stratify=y, random_state=42
)

scaler_imp    = StandardScaler()
X_train_imp_s = scaler_imp.fit_transform(X_train_imp)
X_test_imp_s  = scaler_imp.transform(X_test_imp)

improved = LogisticRegression(max_iter=1000, random_state=42)
improved.fit(X_train_imp_s, y_train_imp)
y_pred_imp = improved.predict(X_test_imp_s)

f1_imp  = f1_score(y_test_imp, y_pred_imp, average="macro")
acc_imp = accuracy_score(y_test_imp, y_pred_imp)

print(f"Baseline accuracy:  {acc_base:.4f}  |  macro-F1: {f1_base:.4f}")
print(f"Improved accuracy:  {acc_imp:.4f}  |  macro-F1: {f1_imp:.4f}")
print(f"Delta F1:           {f1_imp - f1_base:+.4f}")

Baseline accuracy:  0.6368  |  macro-F1: 0.5806
Improved accuracy:  0.6367  |  macro-F1: 0.5805
Delta F1:           -0.0001


In [34]:
import csv

with open("results/metrics.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["model", "accuracy", "macro_f1", "note"])
    writer.writerow(["baseline", acc_base, f1_base,
                     "logistic regression, original features"])
    writer.writerow(["improved", acc_imp, f1_imp,
                     "added dynamic_intensity from Gemini"])

print("Saved results/metrics.csv")

Saved results/metrics.csv


In [35]:
wrong_mask = (y_pred_imp != y_test_imp.values)

wrong = X_test_imp[wrong_mask].copy()
wrong["true_mood"] = y_test_imp[wrong_mask].values
wrong["pred_mood"] = y_pred_imp[wrong_mask]
wrong["genre"]     = df.loc[wrong.index, "track_genre"].values

wrong.sample(20, random_state=42).to_csv(
    "results/wrong_predictions.csv", index=False)

print(f"Total wrong: {wrong_mask.sum()}")
print("Saved results/wrong_predictions.csv")

Total wrong: 5910
Saved results/wrong_predictions.csv
